# Notebook 37 — Serving Open Models with vLLM

    ## Learning objectives

    - Launch and call an OpenAI-compatible vLLM server
- Relate continuous batching and paged KV management to throughput
- Plan capacity, parallelism, structured output, monitoring, and secure deployment

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['httpx>=0.28']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 37.1 Serving changes the optimization target

Local `generate()` is useful for experiments. A server must schedule concurrent requests,
manage variable KV-cache allocations, stream, reject overload, expose health/metrics, and
isolate clients. vLLM combines an optimized engine with HTTP APIs. Continuous batching
admits new sequences as others finish; block-based KV management reduces fragmentation.


## 37.2 Start on a supported accelerator host

Install vLLM according to the current accelerator-specific instructions, then run:

```bash
export HF_TOKEN=hf_your_token
export VLLM_API_KEY=replace-me
vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --host 0.0.0.0 --port 8000 --api-key "$VLLM_API_KEY" --dtype auto
```

Do not expose the port directly to the internet. Put authentication, TLS, request/token
limits, rate limits, and network controls in front of it. Pin the model revision, engine
version, tokenizer, and chat template.


In [ ]:
import os, httpx
base = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
key = os.getenv("VLLM_API_KEY", "local-dev-key")
model = os.getenv("VLLM_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
try:
    response = httpx.post(f"{base}/chat/completions",
        headers={"Authorization": f"Bearer {key}"},
        json={"model": model, "messages": [{"role": "user", "content": "Define GQA briefly."}],
              "temperature": 0, "max_tokens": 100}, timeout=30)
    response.raise_for_status()
    print(response.json()["choices"][0]["message"]["content"])
except Exception as exc:
    print("Start the vLLM server, then rerun:", type(exc).__name__)


## 37.3 Structured output and APIs

Current vLLM releases support OpenAI-compatible chat/completions plus health, model, and
Prometheus metrics endpoints. Structured constraints are passed in the current
`structured_outputs` field; older `guided_json` examples are obsolete. Model chat/tool/
reasoning support depends on templates and parsers. Validate responses in application code.


In [ ]:
schema = {"type": "object", "properties": {
    "term": {"type": "string"}, "definition": {"type": "string"}},
    "required": ["term", "definition"], "additionalProperties": False}
payload_extension = {"structured_outputs": {"json": schema}}
print(payload_extension)


## 37.4 Scale only after measuring

Tensor parallelism shards a model across GPUs; data parallelism replicates it for more
traffic; pipeline/expert parallelism address other model/topology constraints. Communication
can erase gains. Model weights, KV cache, runtime workspace, and fragmentation must fit.
Benchmark TTFT, inter-token latency, p95/p99, tokens/sec, requests/sec, queueing, errors,
cache utilization, and quality with realistic input/output lengths and concurrency.

Scrape `/metrics`, configure readiness around model loading, bound admission queues, canary
upgrades, and test rollback. Quantization and kernel changes require quality regression tests.


## 37.5 vLLM engine mental model

Requests pass through an API frontend/tokenizer into an engine scheduler. Prefill computes prompt
states; decode advances active sequences token by token. A block manager allocates logical KV-cache
blocks to physical memory, reducing fragmentation and enabling operations such as prefix sharing.
Continuous batching changes the active batch each iteration. Sampling, stop conditions, streaming,
and request cancellation interact with scheduler state.

The model's architecture determines tensor shapes, KV heads, supported precision/quantization,
multimodal inputs, and tool/reasoning parsers. The engine version determines kernels and API
behavior. OpenAI compatibility is a client protocol surface, not proof every OpenAI parameter or
model capability behaves identically. Read the current compatibility/serve help for the installed
release and pin it.


In [ ]:
# Capacity worksheet for weights plus KV cache (planning estimate only).
def serving_memory_gib(params_b, weight_bits, layers, kv_heads, head_dim,
                       cached_tokens, kv_bits=16, workspace_fraction=.15):
    weights = params_b * 1e9 * weight_bits/8
    kv = 2 * layers * kv_heads * head_dim * cached_tokens * kv_bits/8
    subtotal = weights + kv
    return {"weights":weights/2**30, "kv":kv/2**30,
            "with_workspace_margin":subtotal*(1+workspace_fraction)/2**30}
print(serving_memory_gib(7, 16, 32, 8, 128, cached_tokens=32_768*8))
print("cached_tokens is total across concurrent sequences, not max context alone.")


## 37.6 Server configuration and compatibility

Important controls include served model name/revision, tokenizer/chat template, dtype,
quantization, maximum model length, GPU memory utilization, maximum batched tokens/sequences,
prefix caching, speculative decoding, tensor/data/pipeline/expert parallelism, tool/reasoning
parser, structured-output backend, and logging/metrics. Defaults evolve. Save the exact command or
YAML config and `vllm --version`; CLI flags override config according to documented precedence.

A chat endpoint needs a valid chat template. Tool calls require model formatting plus a matching
parser and application loop. Reasoning models may need a reasoning parser. Structured output
constrains generation but can reduce throughput or reject unsupported schemas. Quantized formats
need compatible kernels/hardware and quality evals. LoRA serving changes memory/routing and dynamic
adapter loading is a security/operations decision, not a default internet-facing feature.


In [ ]:
# Produce a reviewable serve command from explicit settings—do not execute in this notebook.
config = {
    "model":"Qwen/Qwen2.5-1.5B-Instruct", "host":"127.0.0.1", "port":8000,
    "dtype":"auto", "max-model-len":8192, "gpu-memory-utilization":0.90,
    "enable-prefix-caching":True,
}
command = ["vllm", "serve", config.pop("model")]
for key, value in config.items():
    if value is True: command.append(f"--{key}")
    elif value is not False: command.extend([f"--{key}", str(value)])
print(" ".join(command))


## 37.7 Parallelism, replicas, and topology

Tensor parallelism shards layer matrices and introduces frequent collectives; keep it within a
fast-connected node when possible. Pipeline parallelism splits layers/stages but can introduce
bubbles. Data parallelism replicates the model and distributes requests, increasing aggregate
throughput and fault isolation when each replica fits. Expert parallelism distributes MoE experts.
Multi-node setups need explicit launch/network/NCCL/Ray or other executor configuration and failure
handling. More GPUs can be slower when communication dominates.

Choose the smallest parallel group that makes one replica meet single-request latency and memory,
then add replicas for traffic when possible. Load balancing should consider queue/cache locality,
not blind round robin. Prefix-cache benefits disappear if identical prefixes scatter. Autoscale on
queueing/token load with enough model-load lead time. Keep spare capacity for failures and deploys.
Benchmark topology with real prompt/output lengths and concurrency.


In [ ]:
# Simple throughput/capacity planning table.
offered_rps = 4
avg_input, avg_output = 800, 200
offered_tokens = offered_rps * (avg_input + avg_output)
for replicas in [1,2,4]:
    capacity_per_replica = 1800  # measured total tokens/s example
    utilization = offered_tokens / (replicas*capacity_per_replica)
    print(replicas, f"offered utilization={utilization:.1%}",
          "headroom_ok" if utilization < .7 else "queue risk")


## 37.8 Benchmark methodology and optimization

Separate offline throughput from online latency benchmarks. Use representative prompt/output
length distributions and arrival patterns. Warm up. Report TTFT, inter-token latency, end-to-end
p50/p95/p99, input/output tokens/sec, request throughput, queue time, errors, aborts, GPU utilization,
memory/KV occupancy, and prefix hit rate. Confirm generated quality and token counts; a configuration
that truncates or emits shorter answers appears artificially fast.

Optimization order: establish correctness; size context/admission; choose supported dtype/kernels;
enable prefix caching for repeated prefixes; tune batching limits against tail latency; evaluate
quantization; consider speculative decoding when draft acceptance and workload justify it; then
evaluate parallelism/replicas. Change one factor at a time. CUDA graphs/compilation and kernel
selection can have shape-dependent tradeoffs. Record power/cost if economics matter.


## 37.9 Production and security reference

Put the server behind authenticated TLS ingress with per-tenant rate/token/request-size limits.
Bind admin/dev endpoints privately. Do not use a shared example API key. Control model/tokenizer
downloads, remote code, adapter loading, filesystem/cache permissions, and network egress. Prompts
and outputs are sensitive; configure logs intentionally. Patch engine/model dependencies and scan
images. Validate structured/tool output in the application.

Readiness should wait for loaded/warmed service; liveness should not restart merely because the GPU
is busy. Scrape metrics and correlate request IDs. Graceful shutdown drains/aborts according to
policy. Test OOM recovery, worker/GPU failure, corrupt request, overload, client cancellation,
rolling deploy, model-load failure, and rollback. Pin image digest, vLLM version, model commit,
tokenizer/template, config, driver/CUDA, and hardware in the release record.

Colab is suitable for client experiments and some temporary single-GPU exploration, not a durable
multi-user vLLM service. Use controlled Linux GPU infrastructure for serving benchmarks and
production conclusions.


## 37.10 vLLM operations reference

| Question | Evidence to collect |
|---|---|
| Does it fit? | Weights + total concurrent KV + workspace/fragmentation margin |
| Is it fast? | TTFT, ITL, end-to-end percentiles, tokens/sec under realistic load |
| Does it preserve quality? | Pinned-model eval after dtype/quantization/kernel/config changes |
| Does it scale? | Per-topology throughput, communication, queueing, failure behavior |
| Is it compatible? | Chat template, tools/reasoning parser, structured schema, modalities |
| Is it operable? | health/readiness, metrics, drains, canary/rollback, OOM/failure tests |

Pin vLLM/image digest, model and tokenizer commits, template, serve config, driver/CUDA, and hardware.
Use current CLI/API docs because flags and structured/tool behavior evolve. OpenAI-compatible means a
protocol surface, not identical semantics/features.

Protect ingress with authenticated TLS, per-tenant token/concurrency limits, payload bounds, and private
admin endpoints. Control remote code and adapter loading. Monitor total cached tokens/KV pressure,
queue time, prefix hit rate, aborts/errors, and GPU health—not GPU utilization alone.


## Exercises

    1. Serve a small model and capture TTFT at concurrency 1, 4, and 16.
2. Estimate model plus KV-cache memory for a target workload.
3. Add schema-constrained extraction and validate it with Pydantic.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
